# Week 4 · Part 1 — How three classifiers decide

**Question:** a decision tree, KNN, and an SVM see the same points. How does each one draw the line between two classes?

| Word | Meaning |
|---|---|
| decision boundary | the line where a model's prediction switches from one class to the other |
| Gini impurity | how mixed the classes are in a group of rows: 0 = one class only |
| neighbour | a training row close to the new row |
| margin | the empty "street" an SVM keeps between the classes |
| hyperparameter | a setting we choose before fitting: tree depth, number of neighbours, C |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyClassifier                  # baseline: always the most frequent class
from sklearn.tree import DecisionTreeClassifier, plot_tree # yes/no questions about one input at a time
from sklearn.neighbors import KNeighborsClassifier         # vote of the k nearest training rows
from sklearn.svm import SVC                                # widest possible margin between classes
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_score
from sklearn.datasets import make_moons

## 1. Practice data with two inputs
`make_moons` creates two interleaving half-moons with noise. With only two inputs we can **draw** what each model does.

In [ ]:
X_all, y_all = make_moons(n_samples=300, noise=0.3, random_state=42)
X = pd.DataFrame(X_all, columns=["x1", "x2"])
y = pd.Series(y_all, name="class")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
print("training rows:", len(X_train), " test rows:", len(X_test))
print("class counts in training:", y_train.value_counts().sort_index().to_dict())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(X_train["x1"], X_train["x2"], c=y_train, cmap="coolwarm", edgecolor="k", s=18)
ax.set(title="Training rows", xlabel="x1", ylabel="x2"); plt.show()

**What we see:** two classes of about the same size that curl into each other. No straight line separates them perfectly.

The helper below colours the region where a fitted model predicts each class. The border between the colours is the **decision boundary**.

In [ ]:
def plot_boundary(model, X, y, title, ax):
    """Colour the regions where a fitted model predicts each class."""
    c0, c1 = X.columns
    g0, g1 = np.meshgrid(np.linspace(X[c0].min(), X[c0].max(), 300),
                         np.linspace(X[c1].min(), X[c1].max(), 300))
    grid = pd.DataFrame({c0: g0.ravel(), c1: g1.ravel()})
    ax.contourf(g0, g1, model.predict(grid).reshape(g0.shape), alpha=.25, cmap="coolwarm")
    ax.scatter(X[c0], X[c1], c=y, cmap="coolwarm", edgecolor="k", s=15)
    ax.set(title=title, xlabel=c0, ylabel=c1)

## 2. Decision tree: yes/no questions about one input
A tree asks questions such as "is x2 ≤ 0.40?". At each step it picks the question that makes the groups **least mixed**, measured by Gini impurity (Lecture 4, slides 13–17):

$$\text{Gini} = 1 - \sum_i p_i^2$$

where $p_i$ is the share of class $i$ in the group. Gini is 0 for a pure group and 0.5 for a 50/50 mix of two classes.

**New tool — `DecisionTreeClassifier(max_depth=3)`**: at most three questions from the top to any leaf.

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(12, 5))
plot_tree(tree, feature_names=list(X.columns), filled=True, ax=ax); plt.show()

**What we see:** the first question is about `x2`. Each box shows the question, its Gini, the number of rows, and the class counts.

Check the Gini of the top box (the **root**) by hand, and the lecture example from slide 16 (80 vs 20 rows):

In [ ]:
p = y_train.value_counts(normalize=True)
print("root Gini by hand:", round(1 - (p ** 2).sum(), 4), "  tree:", round(tree.tree_.impurity[0], 4))
print("lecture example 80/20:", round(1 - (0.8 ** 2 + 0.2 ** 2), 2))

A split is judged by the **weighted** Gini of the two child groups (slide 14): each child's Gini, weighted by its share of the rows.

In [ ]:
t = tree.tree_
left, right = t.children_left[0], t.children_right[0]
n_left, n_right = t.n_node_samples[left], t.n_node_samples[right]
weighted = (n_left * t.impurity[left] + n_right * t.impurity[right]) / (n_left + n_right)
print("left:", n_left, "rows, Gini", round(t.impurity[left], 3))
print("right:", n_right, "rows, Gini", round(t.impurity[right], 3))
print("weighted Gini after the first split:", round(weighted, 3))

**What we see:** one question lowers impurity from 0.500 to 0.286. The tree chose `x2 ≤ 0.40` because no other question lowered it more.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_boundary(tree, X_train, y_train, "Decision tree, depth 3", ax); plt.show()

**What we see:** the boundary is made of horizontal and vertical steps, because every question looks at **one input at a time**.

## 3. KNN: ask the nearest neighbours
KNN does not learn rules. It **stores** the training rows. To predict a new row it finds the $k$ nearest rows by Euclidean distance (slide 23) and takes a **majority vote** (slides 26–27).

**New tool — `KNeighborsClassifier(n_neighbors=5)`.** First the same vote by hand for one test row near the border between the moons:

In [ ]:
new_row = X_test.iloc[10]
distance = np.sqrt(((X_train - new_row) ** 2).sum(axis=1))
nearest = pd.DataFrame({"distance": distance, "class": y_train}).nsmallest(5, "distance")
print("new row:", new_row.round(3).to_dict())
display(nearest.round(3))
print("vote:", nearest["class"].value_counts().to_dict())

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5).fit(X_train, y_train)
print("KNN predicts:", knn.predict(X_test.iloc[[10]])[0], "  actual class:", y_test.iloc[10])

**What we see:** three neighbours say class 1, two say class 0, so KNN predicts 1. `predict` returns the class that won the vote. That is all KNN does.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_boundary(knn, X_train, y_train, "KNN, k = 5", ax); plt.show()

**What we see:** the boundary follows the local shape of the data and can be irregular.

## 4. SVM: the widest street between the classes
A linear SVM looks for the line with the **widest margin** between the classes (slides 30–32). Rows on or inside the margin are the **support vectors**; they alone decide where the line goes. `C` sets how strictly errors are punished: small `C` allows more rows inside the margin.

With the **RBF kernel** (slide 35) the boundary may curve.

**New tool — `SVC(kernel="linear")` and `SVC(kernel="rbf")`.**

In [ ]:
svm_linear = SVC(kernel="linear", C=1).fit(X_train, y_train)
svm_rbf = SVC(kernel="rbf", C=1).fit(X_train, y_train)
print("support vectors, linear:", svm_linear.n_support_.sum(), "  RBF:", svm_rbf.n_support_.sum())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_boundary(svm_linear, X_train, y_train, "SVM, linear kernel", axes[0])
plot_boundary(svm_rbf, X_train, y_train, "SVM, RBF kernel", axes[1]); plt.show()

**What we see:** the linear SVM can only draw a straight line. The RBF kernel draws a smooth curve that follows the moons. About a third of the rows are support vectors because the classes overlap.

## 5. Complexity: when a model memorises
Each model has a setting that controls how flexible it is: `max_depth`, `n_neighbors`, `C`. A model that is too flexible fits the training rows perfectly but does worse on new rows: **overfitting** (Lecture 2).

We compare mean training accuracy with mean cross-validated accuracy, using the same stratified folds as in Week 2.

In [ ]:
def train_vs_cv(model):
    """Mean training accuracy and mean validation accuracy over the five folds."""
    r = cross_validate(model, X_train, y_train, cv=cv, return_train_score=True)
    return round(r["train_score"].mean(), 3), round(r["test_score"].mean(), 3)

In [ ]:
rows = []
for d in [1, 2, 3, 5, 10, None]:
    train_acc, cv_acc = train_vs_cv(DecisionTreeClassifier(max_depth=d, random_state=42))
    rows.append({"max_depth": str(d), "train accuracy": train_acc, "CV accuracy": cv_acc})
pd.DataFrame(rows)

In [ ]:
rows = []
for k in [1, 3, 5, 15, 51]:
    train_acc, cv_acc = train_vs_cv(KNeighborsClassifier(n_neighbors=k))
    rows.append({"n_neighbors": k, "train accuracy": train_acc, "CV accuracy": cv_acc})
pd.DataFrame(rows)

**What we see:**
- A tree without a depth limit is 100% right on training rows but only 84% on validation rows. Depth 2–3 is best (about 90%).
- KNN with k = 1 is also 100% on training rows: each row is its own nearest neighbour.
- The gap between training and validation accuracy is the sign of overfitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_boundary(DecisionTreeClassifier(random_state=42).fit(X_train, y_train), X_train, y_train, "Tree, no depth limit", axes[0])
plot_boundary(KNeighborsClassifier(n_neighbors=1).fit(X_train, y_train), X_train, y_train, "KNN, k = 1", axes[1]); plt.show()

**What we see:** both boundaries twist around single points. They describe noise in the training rows, not the moons.

## 6. Summary

| | Decision tree | KNN | SVM |
|---|---|---|---|
| How it decides | yes/no questions, one input at a time | vote of the k nearest rows | widest margin; kernel lets it curve |
| Complexity setting | `max_depth` | `n_neighbors` (small k = flexible) | `C`, kernel |
| Boundary | steps | irregular, local | straight or smooth curve |
| Uses distances? | no | yes | yes |

KNN and SVM measure distances between rows. Part 2 shows why that makes the **scale** of each input important.

**Sources:** AITU Lecture 4, slides 4–35; ISLP, Chapter 8.1 (trees) and Chapter 9 (SVM); scikit-learn user guide.

## Extension: KNN for regression (Week 3 data)
KNN can also predict a number: the **average** of the neighbours' prices (Lecture 4, slides 25 and 28). We compare it with the Week 3 linear models on the same five folds. Distances need inputs on one scale, so KNN gets a `StandardScaler` (Part 2 explains why). This needs `Real Estate.csv` from Week 3.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold

houses = pd.read_csv("Real Estate.csv")
inputs = ["transaction_date_decimal", "house_age_years", "distance_to_mrt_m",
          "convenience_stores", "latitude", "longitude"]
h_train, h_test = train_test_split(houses, test_size=0.20, random_state=42)
folds = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_mae(model):
    s = -cross_val_score(model, h_train[inputs], h_train["price_10k_twd_per_ping"], cv=folds,
                         scoring="neg_mean_absolute_error")
    return round(s.mean(), 3)

print("linear regression, six inputs:", cv_mae(LinearRegression()))
for k in [5, 10, 20]:
    print(f"KNN regression, k = {k}:", cv_mae(make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k))))

**What we see:** KNN (MAE about 6.0–6.2) beats the straight-line model (6.55) without being told that price falls along a curve: it simply averages nearby sales. In Week 3 the linear model with log distance reached 5.56 and still wins, and it can be explained with coefficients. Flexibility helps; a well-chosen simple model can help more.